In [4]:
import os 
os.environ['OPENAI_API_KEY'] = '3'

In [7]:
%pip install -q youtube-transcript-api langchain langchain-community langchain-openai faiss-cpu tiktoken python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: C:\Users\Abish\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [9]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter, TextSplitter
from langchain_openai import OpenAIEmbeddings,ChatOpenAI
from langchain_core.prompts import PromptTemplate


In [ ]:
video_id = 'RJFX4fQpxbA'  # Replace with your YouTube video ID
try:
    transcript_list = YouTubeTranscriptApi.get_transcript(video_id,languages=['en'])
    transcript = " ".join([chunk['text'] for chunk in transcript_list])
    print(transcript)
except TranscriptsDisabled:
    print("Transcripts are disabled for this video.")
    


In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])
print(f"Number of chunks created: {len(chunks)}")


In [ ]:
embeddings = OpenAIEmbeddings()
vector_store = FAISS.from_documents(chunks, embeddings)

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})


In [ ]:
retriever.invoke("What is the video about?")


In [ ]:
llm=ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)

In [ ]:
prompt = PromptTemplate(
    template="""
    You are a hepful assistant.
    Answer Only from the provoded transcript context.
    If the context is not sufficient to answer the question, say you don't know.
    {context}
    Question: {question}
    """,
    input_variables=["context", "question"]
)

In [ ]:
question = "What is the video about?"
retrieved_docs = retriever.invoke(question)


In [ ]:
context_text="\n\n".join([doc.page_content for doc in retrieved_docs])

In [ ]:
final_prompt = prompt.format({"context": context_text, "question": question})

In [ ]:
answer = llm.invoke(final_prompt)
print("Answer:", answer.content)

In [ ]:
from langchain_core.runnables import RunnableLambda,RunnablePassthrough,RunnableParallel,RunnableSequence
from langchain_core.output_parsers import OutputParserException, StrOutputParser, JsonOutputParser


In [ ]:
def format_docs(retrieved_docs):
    context_text="\n\n".join([doc.page_content for doc in retrieved_docs])
    return context_text


In [ ]:
parallel_chain=RunnableParallel(
   {
"context":retriever|RunnableLambda(format_docs),
    "question":RunnablePassthrough()
   } 
)

In [ ]:
question = "What is the video about?"
parallel_chain.invoke({"question": question})

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain|prompt|llm|parser


In [ ]:
main_chain .invoke({"question": question})